# Learned Morphology (AE + Flow) — Integration Tests

Validates that the frozen AutoEncoder + Normalizing Flow (`shine.morphology`)
load correctly, render correctly in isolation, and wire correctly into
`MultiExposureScene` for the 64px stamp tier — **before** running any actual
shear inference. See `MORPHOLOGY_AE_INTEGRATION.md` for the full roadmap.

Sections 1-9 are integration checks (loading, isolated rendering, NumPyro
model tracing, forward-model sanity). Section 10 is a placeholder for the
full inference test (MAP/NUTS + known-shear auto-consistency, roadmap Step 7)
to be added next.

## 0. Setup (Colab)

In [ ]:
# Run this cell only on a fresh Colab runtime.
# Pin flowjax/paramax explicitly -- letting pip resolve them freely against
# the rest of the stack can hit "resolution-too-deep" and never finish.

# !pip install "equinox==0.13.6" "einops>=0.8,<0.9"
# !pip install "flowjax==17.2.1" "paramax==0.0.4"
# !pip install git+https://github.com/GalSim-developers/JAX-GalSim.git

# !git clone <repo-url> SHINE
# %cd SHINE
# !pip install -e ".[test]"

# Upload/mount the AE + flow checkpoint directories and data/EUC_VIS_SWL/
# (git-lfs) before running the cells below.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*complex128.*", module="jax_galsim")

import math
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import numpyro
import numpyro.handlers as handlers
import jax_galsim as galsim

from shine.morphology.loader import load_frozen_autoencoder, load_frozen_flow
from shine.morphology.prior import sample_latent_codes
from shine.morphology.render import render_learned_galaxy
from shine.morphology.config import LearnedMorphologyConfig
from shine.euclid.config import (
    EuclidDataConfig,
    EuclidInferenceConfig,
    SourceSelectionConfig,
)
from shine.euclid.data_loader import EuclidDataLoader
from shine.euclid.scene import MultiExposureScene, render_model_images

%matplotlib inline

# --- Fill these in before running ---
AE_CHECKPOINT_DIR = "/content/checkpoints/ae/epoch_2000"      # TODO
AE_EPOCH = 2000
FLOW_CHECKPOINT_DIR = "/content/checkpoints/flow/epoch_420"   # TODO
FLOW_EPOCH = 420

DATA_DIR = Path("data/EUC_VIS_SWL")  # bundled Euclid VIS test data (git-lfs)

## 1. Load AE + Flow checkpoints

In [ ]:
ae = load_frozen_autoencoder(AE_CHECKPOINT_DIR, AE_EPOCH)
flow = load_frozen_flow(FLOW_CHECKPOINT_DIR, FLOW_EPOCH)

print(f"AE:   nx={ae.nx}, ny={ae.ny}, scale={ae.scale}")
print(f"Flow: latent_dim={list(flow.latent_dim)}, cond_dim={flow.cond_dim}")

assert ae.nx == ae.ny == 64, "unexpected AE stamp size"
assert abs(ae.scale - 0.1) < 1e-6, "AE pixel scale must match EuclidDataConfig.pixel_scale"
assert flow.cond_dim is None, "conditional flows need catalog covariates -- not implemented in sample_latent_codes"


## 2. AutoEncoder sanity checks

`LearnedMorphologyConfig` does **not** cross-check that the flow was trained
on latents from this exact AE checkpoint -- do it here.

In [ ]:
# a) latent shape produced by ae.encode matches what the flow expects
dummy_img = jnp.zeros((1, ae.nx, ae.ny))
z_encoded = ae.encode(dummy_img, key=None)
print("ae.encode output shape:", z_encoded.shape)
assert tuple(z_encoded.shape) == tuple(flow.latent_dim), (
    f"AE latent shape {z_encoded.shape} != flow.latent_dim {tuple(flow.latent_dim)} "
    "-- this AE/flow pair were likely not trained together."
)

# b) dropout truly disabled by inference_mode -> decode must be deterministic across keys
z = jax.random.normal(jax.random.key(0), flow.latent_dim)
g_a = ae.decode(z, key=jax.random.key(0))
g_b = ae.decode(z, key=jax.random.key(1))
assert jnp.array_equal(g_a, g_b), "decode is not deterministic -- dropout may still be active"

# c) decoded image is non-negative (softplus) and finite
print("decode() range:", float(g_a.min()), float(g_a.max()))
assert jnp.all(jnp.isfinite(g_a)) and jnp.all(g_a >= 0)

## 3. Reconstruction on a real Euclid VIS cutout (exploratory)

The AE was trained on `Train-AE`'s own cutouts, not necessarily on Euclid VIS
images directly -- this is a domain/units sanity check (roadmap risk #6:
ADU calibration mismatch), not a formal validation.

In [ ]:
data_dir = Path(DATA_DIR)  # coerce in case DATA_DIR was reassigned as a plain str

# NB: background_paths matters here -- without it, the loader falls back to
# a single sigma-clipped median over the whole quadrant instead of the real
# per-pixel background map, and the residual background structure left in
# the cutout gets "explained" by the AE as extra galaxy flux (typically
# showing up as an over-extended, diffuse reconstruction).
probe_config = EuclidInferenceConfig(
    data=EuclidDataConfig(
        exposure_paths=sorted(str(p) for p in data_dir.glob("EUC_VIS_SWL-DET-*_3-4-F.fits.gz")),
        psf_path=str(data_dir / "PSF_3-4-F.fits.gz"),
        catalog_path=str(data_dir / "catalogue_3-4-F.fits.gz"),
        background_paths=sorted(str(p) for p in data_dir.glob("EUC_VIS_SWL-BKG-*_3-4-F.fits.gz")),
    ),
    sources=SourceSelectionConfig(max_sources=20, min_snr=50.0),
)
probe_data = EuclidDataLoader(probe_config).load()

# Scan sources for the first one with a full, non-edge-truncated 64x64
# cutout. The PSF stamp does NOT need to match ae.nx/ae.ny -- galsim wraps
# galaxy and PSF as independent InterpolatedImages, so a smaller PSF stamp
# (Euclid's default is 21x21, see EuclidPSFModel.stamp_size) is expected
# and fine.
half = ae.nx // 2
image0 = np.asarray(probe_data.images[0])
cutout, psf_img = None, None
for i in range(probe_data.n_sources):
    pos = np.asarray(probe_data.pixel_positions[i, 0, :]).round().astype(int)
    y0, y1, x0, x1 = pos[1] - half, pos[1] + half, pos[0] - half, pos[0] + half
    if y0 < 0 or x0 < 0 or y1 > image0.shape[0] or x1 > image0.shape[1]:
        continue
    cutout = image0[y0:y1, x0:x1]
    psf_img = np.asarray(probe_data.psf_images[i, 0])
    print(f"Using source {i}/{probe_data.n_sources}: cutout shape={cutout.shape}, psf shape={psf_img.shape}")
    break

if cutout is not None and cutout.shape == (ae.nx, ae.ny):
    z_real = ae.encode(cutout[None], key=None)
    g_clean = ae.decode(z_real, key=None)  # PSF-deconvolved "clean" profile -- not directly
                                            # comparable to `cutout` (which still has the real PSF in it)
    recon = np.asarray(ae.convolve(g_clean, psf_img[None])[0])  # re-convolve w/ the *same* real PSF, for a fair comparison

    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    for ax, im, title in zip(
        axes,
        [cutout, np.asarray(g_clean[0]), recon, cutout - recon],
        ["observed (PSF-convolved)", "AE decode() -- deconvolved", "reconvolved w/ real PSF", "residual"],
    ):
        ax.imshow(im, origin="lower")
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.show()
else:
    print(f"No source among the first {probe_data.n_sources} had a full unclipped "
          f"{ae.nx}x{ae.ny} cutout -- raise max_sources above or widen the search.")

### 3b. Saturation check -- is the encoder out of its trained input range?

`Saturate("softclip2")` clamps the latent code to ±5 (`nn/layers.py`). If
`z_real` sits right at that boundary in every dimension while typical prior
draws (`flow.sample()`) don't, the encoder is almost certainly seeing inputs
far outside its trained value range (ADU/normalization mismatch, roadmap
risk #6) -- the reconstruction failure is then an input-calibration issue,
not a bug in the AE/render pipeline itself.

In [ ]:
print("cutout stats: min=%.2f max=%.2f mean=%.2f" % (cutout.min(), cutout.max(), cutout.mean()))
print("z_real (post-saturation):", np.asarray(z_real).ravel())
print("|z_real| max:", float(jnp.max(jnp.abs(z_real))))

z_prior_probe = flow.unflatten_latent(flow.sample(key=jax.random.key(99), sample_shape=(200,)))
print("typical |z| over 200 prior draws: max=%.3f, 99th pct=%.3f" % (
    float(jnp.max(jnp.abs(z_prior_probe))),
    float(jnp.percentile(jnp.abs(z_prior_probe), 99)),
))
# If |z_real| clusters near 5.0 while the prior draws don't -> saturation
# confirmed: cutout needs the same normalization Train-AE applied at
# training time before being passed to ae.encode (check Train-AE's own
# data-loading/preprocessing code for the exact transform).

### 3c. Flux-scale check: does `decode(z)` even live in ADU units?

`ae.encode` saturating on raw Euclid ADU strongly suggests `Train-AE` trained
on cutouts in a different flux scale (normalized, stretched, or just a
different survey/units). If so, `ae.decode(z)` for an **unsaturated**,
in-distribution `z` (drawn from the flow prior, not encoded from a real
image) is probably *not* in ADU either -- and `render_learned_galaxy` feeds
that array directly into the pixel likelihood as if it were ADU. This is the
one to resolve before trusting any inference result.

In [ ]:
z_unsat = flow.unflatten_latent(flow.sample(key=jax.random.key(7), sample_shape=(50,)))
decoded = jax.vmap(lambda zi: ae.decode(zi, key=None))(z_unsat)[:, 0]

print("decode(z~prior) pixel range : min=%.3f max=%.3f mean=%.3f" % (
    float(decoded.min()), float(decoded.max()), float(decoded.mean())
))
print("decode(z~prior) per-image peak (median over 50 draws): %.3f" % (
    float(jnp.median(jnp.max(decoded, axis=(1, 2))))
))
print("decode(z~prior) per-image total flux (median over 50 draws): %.3f" % (
    float(jnp.median(jnp.sum(decoded, axis=(1, 2))))
))

print("\nFor comparison, real Euclid VIS cutout (source 0):")
print("  peak=%.1f ADU, sum=%.1f ADU" % (float(cutout.max()), float(cutout.sum())))
print("  catalog_flux_adu range: [%.1f, %.1f] ADU" % (
    float(probe_data.catalog_flux_adu.min()), float(probe_data.catalog_flux_adu.max())
))
# A gap of orders of magnitude here (not just a factor ~2-5) confirms
# decode() is not in ADU -- Train-AE's exact input normalization needs to
# be found and inverted (or matched) before render_learned_galaxy's output
# can be trusted as a physical ADU profile.

## 4. Flow sanity checks

In [ ]:
z_flat = flow.sample(key=jax.random.key(0), sample_shape=(500,))
print("per-dimension std (mode-collapse check):", jnp.std(z_flat, axis=0))

log_p = flow.log_prob(z_flat)
assert jnp.all(jnp.isfinite(log_p)), "non-finite log_prob -- degenerate flow"

# sample_latent_codes pushes a Normal(0,1) base through flow.forward -- check
# that this matches the flow's own generative process (flow.sample).
z_base = jax.random.normal(jax.random.key(1), (500, math.prod(flow.latent_dim)))
z_via_forward = jax.vmap(flow.forward)(z_base)

n_dims_to_plot = min(4, z_flat.shape[-1])
fig, axes = plt.subplots(1, n_dims_to_plot, figsize=(3 * n_dims_to_plot, 3))
axes = np.atleast_1d(axes)
for d, ax in enumerate(axes):
    ax.hist(np.asarray(z_flat[:, d]), bins=30, alpha=0.5, label="flow.sample")
    ax.hist(np.asarray(z_via_forward[:, d]), bins=30, alpha=0.5, label="forward(base)")
    ax.set_title(f"dim {d}")
axes[0].legend()
plt.tight_layout()
plt.show()

# The bijection must be genuinely invertible (it's what makes this a valid flow).
bijection = flow.flow.bijection
z_round_trip = bijection.inverse(bijection.transform(z_base[0]))
assert jnp.allclose(z_round_trip, z_base[0], atol=1e-4), "bijection is not accurately invertible"

## 5. AE + Flow composed — prior-predictive mosaic

In [ ]:
z_mosaic = flow.unflatten_latent(flow.sample(key=jax.random.key(2), sample_shape=(25,)))
imgs = jax.vmap(lambda zi: ae.decode(zi, key=None))(z_mosaic)[:, 0]

fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for im, ax in zip(imgs, axes.flat):
    ax.imshow(np.asarray(im), origin="lower")
    ax.axis("off")
fig.suptitle("Prior-predictive galaxies: z ~ flow, g = AE.decode(z)")
plt.show()

assert jnp.all(jnp.isfinite(imgs))
# Eyeball: 25 visibly different morphologies (no mode collapse), no NaNs/artifacts.

## 6. NumPyro reparametrization (`sample_latent_codes`)

In [ ]:
def _z_model(n):
    with numpyro.plate("sources", n):
        return sample_latent_codes("z", flow, n)

z_trace = handlers.trace(handlers.seed(lambda: _z_model(8), jax.random.PRNGKey(0))).get_trace()

print("z_base shape:", z_trace["z_base"]["value"].shape)
print("z shape:     ", z_trace["z"]["value"].shape)
assert z_trace["z_base"]["value"].shape == (8, math.prod(flow.latent_dim))
assert z_trace["z"]["value"].shape == (8, *flow.latent_dim)
assert z_trace["z"]["type"] == "deterministic"

## 7. `render_learned_galaxy` in isolation (real weights)

In [ ]:
gsparams = galsim.GSParams(minimum_fft_size=64, maximum_fft_size=64)
flat_psf = jnp.zeros((ae.nx, ae.ny)).at[ae.nx // 2, ae.ny // 2].set(1.0)
identity_wcs = jnp.array([ae.scale, 0.0, 0.0, ae.scale])

z0 = z_mosaic[0]  # reuse a sample from the prior-predictive mosaic above

base = render_learned_galaxy(
    z0, 0.0, 0.0, flat_psf, identity_wcs, 0.0, 0.0, True, ae, ae.nx, ae.scale, gsparams
)
sheared = render_learned_galaxy(
    z0, 0.05, 0.02, flat_psf, identity_wcs, 0.0, 0.0, True, ae, ae.nx, ae.scale, gsparams
)
invisible = render_learned_galaxy(
    z0, 0.0, 0.0, flat_psf, identity_wcs, 0.0, 0.0, False, ae, ae.nx, ae.scale, gsparams
)

assert jnp.all(jnp.isfinite(base)) and jnp.all(jnp.isfinite(sheared))
assert not jnp.allclose(base, sheared), "g1/g2 had no effect on the rendered stamp"
assert jnp.array_equal(invisible, jnp.zeros_like(invisible)), "invisible source was not zeroed"

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(np.asarray(base), origin="lower")
axes[0].set_title("g=(0, 0)")
axes[1].imshow(np.asarray(sheared), origin="lower")
axes[1].set_title("g=(0.05, 0.02)")
for ax in axes:
    ax.axis("off")
plt.show()

## 8. Wire into `MultiExposureScene` and trace the NumPyro model

In [ ]:
data_dir = Path(DATA_DIR)  # coerce in case DATA_DIR was reassigned as a plain str

learned_config = EuclidInferenceConfig(
    data=EuclidDataConfig(
        exposure_paths=sorted(str(p) for p in data_dir.glob("EUC_VIS_SWL-DET-*_3-4-F.fits.gz")),
        psf_path=str(data_dir / "PSF_3-4-F.fits.gz"),
        catalog_path=str(data_dir / "catalogue_3-4-F.fits.gz"),
        background_paths=sorted(str(p) for p in data_dir.glob("EUC_VIS_SWL-BKG-*_3-4-F.fits.gz")),
    ),
    sources=SourceSelectionConfig(max_sources=20, min_snr=50.0, exclude_point_sources=False),
    galaxy_stamp_sizes=[64],  # force every selected source onto the learned tier
    learned_morphology=LearnedMorphologyConfig(
        enabled=True,
        ae_checkpoint_dir=AE_CHECKPOINT_DIR,
        ae_epoch=AE_EPOCH,
        flow_checkpoint_dir=FLOW_CHECKPOINT_DIR,
        flow_epoch=FLOW_EPOCH,
        apply_to_stamp_size=64,
        # decode(z) still contains the fixed reference PSF from training
        # (see shine.morphology.psf_residual) -- built by
        # scripts/build_residual_psf.py, not the full local PSF above.
        psf_residual_path=str(data_dir / "PSF_3-4-F_residual.fits.gz"),
    ),
)

learned_data = EuclidDataLoader(learned_config).load()
scene = MultiExposureScene(learned_config, learned_data)
assert scene.ae is not None and scene.flow is not None
model = scene.build_model()

model_trace = handlers.trace(handlers.seed(model, jax.random.PRNGKey(0))).get_trace(
    observed_data=learned_data.images
)
for site in ("g1", "g2", "z_base", "z"):
    print(f"{site}: shape={model_trace[site]['value'].shape}")
for j in range(learned_data.n_exposures):
    assert f"obs_{j}" in model_trace

n_sources = learned_data.n_sources
assert model_trace["z_base"]["value"].shape == (n_sources, math.prod(flow.latent_dim))
assert model_trace["z"]["value"].shape == (n_sources, *flow.latent_dim)
assert jnp.all(jnp.isfinite(model_trace["z"]["value"]))
print("\nModel trace OK -- no NaNs, all expected sites present.")

## 9. Prior-predictive image vs. real observation (visual scale check)

In [ ]:
z_prior = flow.unflatten_latent(flow.sample(key=jax.random.key(3), sample_shape=(n_sources,)))

# flux/hlr/e1/e2/dx/dy are unused on the learned tier but still required by
# render_model_images's signature -- catalog/zero values are fine here.
prior_pred_images = render_model_images(
    dict(
        g1=jnp.float32(0.0),
        g2=jnp.float32(0.0),
        flux=jnp.asarray(learned_data.catalog_flux_adu),
        hlr=jnp.asarray(learned_data.catalog_hlr_arcsec),
        e1=jnp.zeros(n_sources),
        e2=jnp.zeros(n_sources),
        dx=jnp.zeros(n_sources),
        dy=jnp.zeros(n_sources),
        z=z_prior,
    ),
    learned_data,
    pixel_scale=learned_config.data.pixel_scale,
    stamp_sizes=learned_config.galaxy_stamp_sizes,
    ae=scene.ae,
    learned_tier_idx=0,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(np.arcsinh(np.asarray(learned_data.images[0])), origin="lower", cmap="gray_r")
axes[0].set_title("Observed exposure 0")
axes[1].imshow(np.arcsinh(np.asarray(prior_pred_images[0])), origin="lower", cmap="gray_r")
axes[1].set_title("AE prior-predictive (g=0, z~flow)")
for ax in axes:
    ax.axis("off")
plt.show()

print(
    "Observed peak:", float(learned_data.images[0].max()),
    "| Prior-predictive peak:", float(prior_pred_images[0].max()),
)
# Sanity: the two peaks/dynamic ranges should be of the same order of
# magnitude -- a large mismatch points at an ADU/calibration issue (roadmap
# risk #6), not a shear/inference issue.

## 10. [TODO] Full inference test

Not implemented yet. This is where the MAP/NUTS run on `model` and the
known-shear auto-consistency check (roadmap Step 7: inject a known `g1, g2`,
render through the AE, add noise, re-infer, and check the recovered shear)
will go.

In [ ]:
# TODO: full inference test (MAP/NUTS on `model`) + known-shear
# auto-consistency check, to be added next.